In [ ]:
# %% [markdown]
# # GC×GC-MS Peak Detection Analysis
# 
# Ce notebook utilise l'interface réutilisable pour configurer et exécuter l'analyse de détection de pics sur des données GC×GC-MS.

# %% [markdown]
## Imports et Configuration

# %%
import os
import traceback
import netCDF4 as nc
import h5py
from pathlib import Path
from identification import sample_identification

# Import de l'interface réutilisable
from gcgcms_ui import BaseFileUI, ParameterWidget, create_method_widgets, create_action_widgets
import ipywidgets as widgets
from IPython.display import display

# %% [markdown]
## Classe spécialisée pour l'analyse GC×GC-MS

# %%
class GCGCMSAnalysisUI(BaseFileUI):
    """
    Interface spécialisée pour l'analyse GC×GC-MS.
    Hérite de BaseFileUI pour la gestion des fichiers et ajoute les paramètres spécifiques.
    """
    
    def __init__(self):
        """Initialize the GCMS Analysis UI with default parameters and widgets."""
        # Initialiser la classe de base avec les extensions supportées
        super().__init__(supported_extensions=('.cdf', '.h5'))
        self._setup_gcgcms_parameters()
        self._create_parameter_widgets()
        self._create_gcgcms_widgets()
        self._setup_callbacks()
    
    def _setup_gcgcms_parameters(self):
        """Initialize default analysis parameters specific to GC×GC-MS."""
        # Paramètres publics (configurables via UI)
        self.default_params = {
            'abs_threshold': "0",
            'rel_threshold': "0.005",
            'noise_factor': "1.5",
            'min_persistence': "0.0002"
        }
        
        # Paramètres privés (fixes pour cette UI)
        self._fixed_params = {
            'min_distance': 1,
            'sigma_ratio': 1.6,
            'num_sigma': 10,
            'min_sigma': 1,
            'max_sigma': 30,
            'overlap': 0.5,
            'match_factor_min': 650,
            'cluster': True,
            'min_samples': 4,
            'eps': 3,
            'formated_spectra': True
        }
    
    def _create_parameter_widgets(self):
        """Create parameter input widgets with validation."""
        # Noise factor
        self.noise_factor_param = ParameterWidget(
            "Noise factor",
            self.default_params['noise_factor'],
            "Noise scaling factor used to filter detected peaks. "
            "A peak is retained if its intensity is greater than the maximum intensity multiplied by this factor.",
            validator=lambda x: (True, "") if x >= 0 else (False, "Noise factor must be non-negative")
        )
        
        # Min persistence
        self.min_persistence_param = ParameterWidget(
            "Minimum persistence",
            self.default_params['min_persistence'],
            "Minimum topological persistence threshold that a peak must exceed to be considered a true signal rather than noise.",
            validator=lambda x: (True, "") if x >= 0 else (False, "Minimum persistence must be non-negative")
        )
        
        # Absolute threshold
        self.abs_threshold_param = ParameterWidget(
            "Absolute threshold",
            self.default_params['abs_threshold'],
            "Absolute threshold used to filter detected peaks based on their raw intensity.",
            validator=lambda x: (True, "") if x >= 0 else (False, "Absolute threshold must be non-negative")
        )
        
        # Relative threshold
        self.rel_threshold_param = ParameterWidget(
            "Relative threshold",
            self.default_params['rel_threshold'],
            "Relative threshold used to filter detected peaks based on their relative intensity.",
            validator=lambda x: (True, "") if 0 <= x <= 1 else (False, "Relative threshold must be between 0 and 1")
        )
    
    def _create_gcgcms_widgets(self):
        """Create GC×GC-MS specific widgets."""
        # Title
        self.txt_title = widgets.HTML('<H1>GC×GC-MS Analysis Configuration</H1>')
        
        # Method and mode widgets
        self.w_method, self.w_mode, self.r_method, self.r_mode = create_method_widgets()
        
        # NIST matching
        self.nist = widgets.Checkbox(
            value=True,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        )
        
        # Action widgets
        self.run_button, self.clear_button, self.output = create_action_widgets()
    
    def _setup_callbacks(self):
        """Set up callbacks for interactive widgets."""
        self.run_button.on_click(self._on_button_click)
    
    def _validate_parameters(self):
        """Validate all input parameters."""
        errors = []
        
        # Validate each parameter
        for param in [self.noise_factor_param, self.min_persistence_param, 
                      self.abs_threshold_param, self.rel_threshold_param]:
            is_valid, error_msg = param.validate()
            if not is_valid:
                errors.append(error_msg)
        
        # Validate file selections
        selection_errors, _ = self.validate_selections()
        errors.extend(selection_errors)
        
        return errors
    
    def get_scan_number(self, file_path):
        """Get scan number from file."""
        try:
            if file_path.endswith((".h5", ".H5")):
                with h5py.File(file_path, 'r') as f:
                    return f.attrs['scan_number_size']
            elif file_path.endswith((".cdf", ".CDF")):
                with nc.Dataset(file_path, 'r') as dt:
                    return dt.dimensions['scan_number'].size
            else:
                raise ValueError("Unsupported file format. Please provide a .h5 or .cdf file.")
        except Exception as e:
            raise ValueError(f"Error while reading file {file_path}: {e}")
    
    def get_mod_time(self, file_path):
        """Get modulation time based on scan_number from file."""
        scan_number = self.get_scan_number(file_path)
        
        modulation_times = {
            328125: (1.25, "G0/plasma"),
            540035: (1.7, "exhaled air")
        }
        
        if scan_number in modulation_times:
            mod_time, data_type = modulation_times[scan_number]
            print(f"   Data type: {data_type}")
            return mod_time
        else:
            print(f"   ⚠️  Unknown scan_number: {scan_number}, using default modulation time")
            return None
    
    def save_parameters(self, selected_files, output_path, method, mode, 
                       noise_factor, min_persistence, abs_threshold, rel_threshold, nist):
        """Save the analysis parameters to a file."""
        params = {
            "selected_files": selected_files,
            "method": method,
            "mode": mode,
            "noise_factor": noise_factor,
            "min_persistence": min_persistence,
            "abs_threshold": abs_threshold,
            "rel_threshold": rel_threshold,
            "nist": nist,
            **self._fixed_params  # Add fixed parameters
        }
        
        params_file = os.path.join(output_path, 'analysis_parameters.txt')
        with open(params_file, 'w') as f:
            for key, value in params.items():
                f.write(f"{key}: {value}\n")
        print(f"📂 Parameters saved to '{params_file}'")
    
    def analyze_files(self, selected_files, output_path, method, mode, 
                     noise_factor, min_persistence, abs_threshold, rel_threshold, nist):
        """Run the analysis on the specified files."""
        if not selected_files:
            print("❌ Error: No files selected for analysis.")
            return False
            
        # Save parameters
        self.save_parameters(
            selected_files, output_path, method, mode, noise_factor, 
            min_persistence, abs_threshold, rel_threshold, nist
        )
        
        print(f"\n🔍 Starting analysis of {len(selected_files)} file(s):")
        for i, f in enumerate(selected_files, 1):
            print(f"  {i}. {f}")
        
        successful_analyses = 0
        failed_analyses = 0
        
        for i, full_path in enumerate(selected_files, 1):
            print(f"\n{'='*60}")
            print(f"🔬 Processing file {i}/{len(selected_files)}: {full_path}")
            print(f"{'='*60}")
            
            try:
                path = os.path.dirname(full_path)
                file = os.path.basename(full_path)
                
                mod_time = self.get_mod_time(full_path)
                if mod_time is None:
                    print("   ⚠️ Modulation time not specified, using default value of 1.25 seconds")
                    mod_time = 1.25
                print(f"⏱️  Modulation time: {mod_time} seconds")
                
                print(f"🚀 Starting analysis...")
                result = sample_identification(
                    path, file, output_path, mod_time, method, mode,
                    noise_factor, abs_threshold, rel_threshold,
                    self._fixed_params['cluster'], self._fixed_params['min_distance'],
                    self._fixed_params['min_sigma'], self._fixed_params['max_sigma'],
                    self._fixed_params['sigma_ratio'], self._fixed_params['num_sigma'],
                    self._fixed_params['formated_spectra'], self._fixed_params['match_factor_min'],
                    min_persistence, self._fixed_params['overlap'],
                    self._fixed_params['eps'], self._fixed_params['min_samples'], nist
                )
                
                print(f"✅ Analysis completed successfully!")
                print(f"📊 Result: {result}")
                successful_analyses += 1
                
            except Exception as e:
                print(f"❌ Analysis failed for {full_path}:")
                print(f"   Error: {str(e)}")
                failed_analyses += 1
                if hasattr(e, '__traceback__'):
                    traceback.print_exc()
        
        print(f"\n{'='*60}")
        print(f"📊 ANALYSIS SUMMARY")
        print(f"{'='*60}")
        print(f"✅ Successful: {successful_analyses}")
        print(f"❌ Failed: {failed_analyses}")
        print(f"📈 Success rate: {successful_analyses}/{len(selected_files)} ({100*successful_analyses/len(selected_files):.1f}%)")
        
        return successful_analyses > 0
    
    def _on_button_click(self, b):
        """Handle button click event to start analysis."""
        with self.output:
            self.output.clear_output()
            print("🚀 Initializing GC×GC-MS analysis...")
            
            # Validate parameters
            